In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 从项目根目录或 examples 目录启动均可；统一以项目根目录运行。
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "mtp_initializer").is_dir() and (p / "PROJECT.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 scipykit 项目根目录或 examples 目录启动 notebook")
os.chdir(PROJECT_ROOT)
# 根目录用于本地脚本；父目录用于 import scipykit。同步对子进程生效。
python_paths = [str(PROJECT_ROOT), str(PROJECT_ROOT.parent)]
for path in reversed(python_paths):
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(python_paths + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
)
os.environ["NOTEBOOK_NAME"] = "04_日志主题与格式"
print("项目路径:", PROJECT_ROOT)
print("当前解释器:", sys.executable)


# 日志布局、配色和丰富对象
预设布局：`minimal / compact / source / detailed`。配色主题：`default / pastel / light / mono`。`colors=None` 自动检测终端，`True` 强制彩色，`False` 关闭彩色。文件不受终端主题影响。

In [ ]:
from scipykit.log import *
for preset in PRESETS:
    configure_log(preset=preset, colors=True)
    log.success("布局示例", preset=preset)
for theme in THEMES:
    configure_log(preset="compact", theme=theme, colors=True)
    log.warning("配色示例", theme=theme)

## 自定义格式和单句样式
`format` 使用 Loguru record 字段，如 `{time:HH:mm:ss}`、`{file.path}`、`{function}`、`{line}`、`{extra[scope]}`、`{message}`。`theme` 传字典可覆盖各级别样式。正文默认不解析标记；`markup=True` 启用 Rich 标记语法，`style` 设置整句正文。

In [ ]:
configure_log(
    format="{time:HH:mm:ss} · {level.name} · {extra[scope]}{message}",
    theme={"INFO": "bold cyan", "WARNING": "bold magenta"},
    colors=True,
)
log("[bold green]完成[/bold green]，下一步 [yellow]检查结果[/yellow]", markup=True)
log("本句自定义颜色", style="bold white on blue")
log("[1, 2] 和 {a: 1} 默认按原文打印")
log({"数据": [{"id": i, "score": round(i / 7, 3)} for i in range(4)]})

In [ ]:
from rich.table import Table
from rich.panel import Panel
from rich.text import Text

table = Table(title="指标汇总")
table.add_column("方法")
table.add_column("准确率", justify="right")
table.add_row("基线", "0.82")
table.add_row("改进", "0.91")
log(table)
log(Panel("可以直接传入 Rich 的 Table / Panel 对象", title="说明"))
log(Text("Rich Text 保留颜色和强调", style="bold green"))
# Table/Panel 保留文字布局；其内部颜色转为统一主题，Text 保留细粒度样式。

## 文件与终端使用不同级别/格式
文件默认为 TRACE，终端默认为 INFO；即使终端隐藏 DEBUG，文件也能保存。`rotation/retention/compression/enqueue` 透传给 Loguru 文件 sink。示例同一路径只有一个 sink，重复开启不会重复写入。修改已打开文件的配置前先 `stop_log(path)`。

In [ ]:
configure_log(level="WARNING", preset="source", colors=True)
path = Path("assets/04_日志主题与格式/debug.log")
with record_to(path, level="DEBUG", format="{level.name} | {file.name}:{line} | {message}", mode="w"):
    log.debug("只在文件中出现")
    log.warning("终端和文件都出现")
    log("\033[31m清理输入中的 ANSI\033[0m", level="ERROR")
content = path.read_text()
assert "\x1b" not in content
print(content)
configure_log()

## 使用边界
线程和 asyncio 的 `record_to/log_context` 已做隔离测试。子任务会继承创建时的上下文，需在退出 `record_to` 前等待它们结束。`enqueue=True` 是进程内队列写入功能；独立启动的多个进程不应共用同一个带轮转的文件，建议每进程文件或应用层集中收集。普通 `loguru.logger` 的 handlers 与本模块隔离。